# SQuAD 1.1：Baseline、Prompting 与 LoRA
先按 README 安装环境，选择对应 Python 内核。全部结果由本次运行计算。

## 1. 原始数据、文章划分、模型与预算

In [ ]:
from pathlib import Path
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import pandas as pd
from IPython.display import display, Image
from genqa import engine
from genqa.workflow import (read_config, new_run, prepare_experiment, setup_reproducibility,
                            check_cuda, record_run, evaluate_method, train, evaluate_lora)
from genqa.report import make_report

# Notebook可以从仓库根目录或notebooks目录打开。
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
root, config = read_config(ROOT / "configs/experiment.json")
repro = setup_reproducibility(config)
print(check_cuda(config))
run_dir = new_run(root, config)
print("本次结果目录：", run_dir)

In [ ]:
# 下载四份公开原料并校验；重新做正文/问题去重、近重复清理和文章隔离。
# 然后使用固定Qwen分词器，按ID哈希取2048条长度合格样本，先检查两轮token预算。
prepared = prepare_experiment(root, config, output_dir=run_dir / "data")
display(pd.DataFrame([
    {"split": name, "examples": len(rows), "articles": len({r["article_title"] for r in rows})}
    for name, rows in prepared["splits"].items()
]))
display(pd.DataFrame([{k:v for k,v in prepared["budget"].items()
                       if k in ("n", "epochs", "max_train_tokens", "planned_nonpadding_input_tokens", "planned_supervised_target_tokens")}]))
# 监督token是input的一部分，不能相加；padding和FP16重试会分别记录。

In [ ]:
# 模型权重首次下载后会缓存；每次仍会重新进行推理和微调。
model = engine.load_model(str(prepared["model_dir"]),
                          precision=config["runtime"]["precision"], local_files_only=True)
record_run(run_dir, prepared, repro, next(model.parameters()).dtype)
print(config["model"]["model_id"], "revision:", config["model"]["revision"])

## 2. Baseline：原始零样本提示

In [ ]:
# 不更新权重；输入context和question，真实生成后计算训练/验证/测试集指标。
print(prepared["baseline"]["instruction"])
baseline_results = evaluate_method(model, prepared, run_dir, "Baseline")
display(pd.DataFrame(baseline_results).round(3))

## 3. Prompting：明确短答案指令与三个示例

In [ ]:
# 三条示例固定取自本次训练集的不同文章；验证、测试答案不进入提示。
print(prepared["prompting"]["instruction"])
display(pd.DataFrame(prepared["prompting"]["demonstrations"])[["question", "answer"]])
prompting_results = evaluate_method(model, prepared, run_dir, "Prompting")
display(pd.DataFrame(prompting_results).round(3))
del model
engine.free_memory()

## 4. LoRA：两轮训练与逐轮验证

In [ ]:
# 冻结底座，仅更新LoRA参数；具体r/alpha/目标层/学习率均在配置中。
# 只对答案和EOS计算loss，prompt/padding的labels为-100。
display(pd.DataFrame([config["training"]]))
training = train(prepared, run_dir)
display(pd.DataFrame(training["epochs"]).round(4))
print("实际消耗 input tokens：", training["consumed_nonpadding_input_tokens"])
print("预算上限：", training["max_train_tokens"])

In [ ]:
# 每轮独立载入checkpoint评验证集，预先固定第2轮评train/test；不按test选模型。
lora_results = evaluate_lora(prepared, run_dir)
display(pd.DataFrame(lora_results).round(3))

## 5. 三种方法的结果对比

In [ ]:
metrics = make_report(run_dir)
display(metrics.round(3))
display(metrics.pivot(index="method", columns="phase", values=["EM (%)", "F1 (%)"]).round(3))
display(Image(filename=str(run_dir / "results.png")))
# 固定展示前8题，不按哪种方法成功选择例子。
display(pd.read_csv(run_dir / "test_answers.csv").head(8))
engine.save_json(run_dir / "status.json", {"status": "completed", "smoke": False, "formal_run": True})
print("日志、每题预测、CSV、图表与adapter：", run_dir)